# Sprint 4 — Implementing a GPT Model from Scratch

Nesta demonstração, os **embeddings da Sprint 2** e a **Multi-Head Attention da Sprint 3** são reutilizados na arquitetura GPT. Os parâmetros são inicializados aleatoriamente: **nenhum treinamento foi realizado**.


In [ ]:
from pathlib import Path
import sys
import torch

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.tokenization import create_simple_tokenizer
from src.transformer import LayerNorm, FeedForward, TransformerBlock
from src.gpt import GPTModel
from src.generation import generate_tokens

torch.manual_seed(123)
corpus = (ROOT / "data" / "texto_teste.txt").read_text(encoding="utf-8")
tokenizer = create_simple_tokenizer(corpus)
input_ids = torch.tensor([tokenizer.encode("Um modelo de linguagem")], dtype=torch.long)
print("Input IDs:", input_ids, "formato:", tuple(input_ids.shape))


## 1. Componentes isolados

- `LayerNorm`: normaliza cada vetor de dimensão D;
- `FeedForward`: transforma D → F → D com GELU;
- `TransformerBlock`: duas conexões residuais em sequência, com atenção causal.


In [ ]:
x = torch.randn(2, 4, 32)
norm = LayerNorm(32)
ffn = FeedForward(32, 128)
block = TransformerBlock(32, context_length=12, num_heads=4, ff_hidden_dim=128)
print("Entrada [B,T,D]", tuple(x.shape))
print("LayerNorm", tuple(norm(x).shape))
print("FeedForward", tuple(ffn(x).shape))
print("TransformerBlock", tuple(block(x).shape))


## 2. Modelo GPT

A entrada `[B,T]` passa por embeddings `[B,T,D]`, N blocos, LayerNorm final e Language Model Head, que produz logits `[B,T,V]`. Cada posição prevê **o token seguinte** quando o modelo for treinado.


In [ ]:
model = GPTModel(vocab_size=tokenizer.vocab_size,
                 context_length=12, embedding_dim=32, num_heads=4,
                 num_layers=2, ff_hidden_dim=128, dropout=0.1).eval()
with torch.inference_mode():
    logits = model(input_ids)
print("Entrada:", tuple(input_ids.shape))
print("Logits:", tuple(logits.shape))
print("Parâmetros:", sum(p.numel() for p in model.parameters()))
print("Maior probabilidade na última posição:",
      torch.softmax(logits[0, -1, :], dim=-1).max().item())


## 3. Geração autoregressiva

Calcula logits da última posição, divide por temperatura, filtra top-k e escolhe um Token ID; o novo ID é anexado ao contexto. O procedimento se repete. **O texto não precisa ser coerente sem treino**.


In [ ]:
greedy_ids = generate_tokens(model, input_ids, max_new_tokens=6)
sampled_ids = generate_tokens(model, input_ids, max_new_tokens=6,
                              temperature=1.0, top_k=5, do_sample=True,
                              generator=torch.Generator().manual_seed(123))
print("Greedy IDs:", greedy_ids.tolist())
print("Greedy texto:", tokenizer.decode(greedy_ids[0].tolist()))
print("Top-k IDs:", sampled_ids.tolist())
print("Top-k texto:", tokenizer.decode(sampled_ids[0].tolist()))


## 4. Experimentos

Execute `python experimentos/sprint4_experimentos.py` na raiz do repositório. O script compara variações de D, H, N, F, contexto, dropout e geração, produzindo `resultados_sprint4.md` e `analise_sprint4.md` com **medições reais**.

**Interpretação:** diferenças de parâmetros, dimensões, probabilidades e tempos não demonstram aprendizagem linguística antes da Sprint 5.
